# BFF Soup Visualization Examples

This notebook demonstrates all visualization capabilities provided by the `SoupVisualizer` class.

These visualizations replicate the figures from *Computational Life: How Well-formed, Self-replicating Programs Emerge from Simple Interaction* (Agüera y Arcas et al., 2024).

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from src.bff.visualization import SoupVisualizer
import matplotlib.pyplot as plt

# Enable inline plotting
%matplotlib inline

## Initialize Visualizer

In [ ]:
visualizer = SoupVisualizer()

## 1. Complexity Over Time (Figure 5 - Single Run)

Plots the evolution of high-order entropy across epochs for a single simulation run.

In [ ]:
# Generate example data
history = [
    {'epoch': i, 'high_order_entropy': 0.1 + 0.8 * (1 - np.exp(-i/50))}
    for i in range(200)
]

visualizer.plot_complexity_over_time(history)

## 2. Complexity Distribution (Figure 5 - Multi-Run)

Shows complexity evolution across multiple runs with quantile bands.

In [ ]:
# Generate example data for multiple runs
np.random.seed(42)
num_runs = 50
num_epochs = 200

runs = []
for run in range(num_runs):
    # Each run has different random variation
    noise = np.random.normal(0, 0.1, num_epochs)
    history = [
        {'high_order_entropy': max(0, 0.1 + 0.8 * (1 - np.exp(-i/50)) + noise[i])}
        for i in range(num_epochs)
    ]
    runs.append(history)

visualizer.plot_complexity_distribution(runs)

## 3. Token Statistics (Figure 1)

Dual y-axis plot showing unique token count and complexity over time.

In [ ]:
# Generate example token statistics
history = [
    {
        'epoch': i,
        'unique_tokens': int(100 * (1 + np.log(i + 1))),
        'high_order_entropy': 0.1 + 0.8 * (1 - np.exp(-i/50))
    }
    for i in range(200)
]

visualizer.plot_token_statistics(history)

## 4. Mutation Rate Heatmap (Figure 6)

Shows the distribution of time-to-complexity across different mutation rates.

In [ ]:
# Generate example mutation rate data
mutation_rates = [0.0001, 0.001, 0.005, 0.01, 0.02, 0.05]
epoch_bins = [50, 100, 200, 500, 1000]

results = {}
for i, mr in enumerate(mutation_rates):
    for j, eb in enumerate(epoch_bins):
        # Simulate that medium mutation rates reach complexity faster
        if i in [2, 3]:  # 0.005, 0.01
            count = max(0, 20 - abs(j - 2) * 5)
        else:
            count = max(0, 10 - abs(j - 3) * 3)
        results[(mr, eb)] = count

visualizer.plot_mutation_rate_heatmap(results, mutation_rates, epoch_bins)

## 5. Complexity Histogram (Figure 7)

Compares final complexity distributions across different experimental conditions.

In [ ]:
# Generate example comparison data
np.random.seed(42)

datasets = {
    'Random Soup': np.random.beta(2, 5, 100),
    'With Replication': np.random.beta(5, 2, 100),
    'High Mutation': np.random.beta(3, 3, 100)
}

visualizer.plot_complexity_histogram(datasets, bins=20)

## 6. 2D Spatial Soup (Figure 8)

Visualizes spatial distribution of programs in a 2D grid soup.

In [ ]:
# Create mock 2D grid soup
class MockGridSoup:
    def __init__(self, height=32, width=32, epoch=100):
        self.height = height
        self.width = width
        self.epoch = epoch
        self.programs = []
        
        np.random.seed(42)
        for i in range(height * width):
            # Create simple program object with data attribute
            prog = type('Program', (object,), {
                'data': np.random.randint(0, 256, size=10, dtype=np.uint8)
            })
            self.programs.append(prog)

grid_soup = MockGridSoup()
visualizer.plot_2d_grid(grid_soup)

## Saving Figures

All plot methods support saving to file:

In [ ]:
# Example: Save complexity over time plot
history = [
    {'epoch': i, 'high_order_entropy': 0.1 + 0.8 * (1 - np.exp(-i/50))}
    for i in range(200)
]

# Uncomment to save:
# visualizer.plot_complexity_over_time(history, save_path='complexity_evolution.png')

## Integration with Real Simulations

In practice, you would collect metrics during soup evolution:

```python
from src.bff.soup import Soup
from src.bff.metrics import aggregate_soup_complexity
from src.bff.tokens import TokenAnalyzer

# Run simulation
soup = Soup(size=1000)
analyzer = TokenAnalyzer()
history = []

for epoch in range(200):
    soup.step()
    
    metrics = aggregate_soup_complexity(soup)
    tokens = analyzer.count_unique_tokens(soup)
    
    history.append({
        'epoch': epoch,
        'high_order_entropy': metrics['high_order_entropy'],
        'unique_tokens': tokens
    })

# Visualize
visualizer.plot_complexity_over_time(history)
visualizer.plot_token_statistics(history)
```

## Conclusion

The `SoupVisualizer` class provides comprehensive visualization tools for analyzing BFF soup evolution:

1. **Single-run complexity**: Track entropy evolution
2. **Multi-run distributions**: Compare across experiments with quantile bands
3. **Token statistics**: Monitor token diversity and replication
4. **Parameter exploration**: Heat maps for mutation rate sweeps
5. **Distribution comparison**: Histograms for final complexity
6. **Spatial visualization**: 2D grid soup snapshots

All methods support saving to high-resolution PNG files for publications.